# Export real prediction sets for the teaser's qualitative panel

This is an **export, not a re-run**. Nothing is retrained end-to-end and no features are
re-extracted: the cached CLIP features already on Drive are reused, the last-layer ERM head is
refitted in seconds, and the conformal evaluation is the same `evaluate()` call the grid makes.

`conformal_eval.evaluate` already builds a full per-example `membership` matrix and then throws it
away when it aggregates. `return_examples=True` keeps it, so we can show, for a single real image,
the prediction set each policy actually returns.

**Why it will agree with the paper.** Every random draw inside `evaluate` is seeded off
`split_seed` (the cal/test split, both rho resamplings, and the two uniform streams that randomise
APS), and the ERM head fits with L-BFGS, whose solver ignores `random_state`. The export does not
take that on trust: it looks up the stored CSV row for the same cell and **asserts** that seven
aggregates match. If the panel would contradict Table 1, it raises instead of writing files.

Runtime: a few minutes if the CLIP feature cache is warm. GPU only helps if it has to extract.


## 0. Parameters — **EDIT THESE**

In [ ]:
REPO_URL       = "https://github.com/octadion/vgscp.git"
REPO_BRANCH    = "main"
DRIVE_CACHE    = "/content/drive/MyDrive/vgscp_cache"
WATERBIRDS_URL = "https://nlp.stanford.edu/data/dro/waterbird_complete95_forest2water2.tar.gz"

# The cell the teaser uses. CLIP/Waterbirds under ERM is where marginal calibration fails hardest
# (worst-group coverage 0.509 against Mondrian's 0.854), so it is where a single image shows the
# difference most clearly.
DATASET, BACKBONE, METHOD = "waterbirds", "clip_vitb32", "erm"
SCORE, SPLIT_SEED, TRAIN_SEED, RHO_TEST = "APS", 0, 0, 0.95

# The ablation CSV the export verifies itself against. It is searched for in the usual places and,
# failing that, you are prompted to upload it (it is ~23 MB).
CSV_NAME = "calibration_ablation_4bb.csv"

import os, sys, time, subprocess
def sh(cmd, **kw):
    print("$", cmd); return subprocess.run(cmd, shell=True, **kw)

## 1. Install

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
subprocess.run("pip -q install open_clip_torch ftfy regex tqdm pyyaml scikit-learn scipy pandas "
               "matplotlib torchvision", shell=True)

## 2. Mount Drive, clone the repo, point at the cached features

In [ ]:
from google.colab import drive
drive.mount("/content/drive"); os.makedirs(DRIVE_CACHE, exist_ok=True)
REPO_DIR = "/content/vgscp"
sh(f"rm -rf {REPO_DIR} && git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)

from study_robust_train.colab_data import prepare_waterbirds
os.environ["WATERBIRDS_ROOT"] = prepare_waterbirds(DRIVE_CACHE, WATERBIRDS_URL)

# Same symlinks as the ablation notebook, so this hits the caches the paper's numbers came from
# instead of re-extracting features.
for c in ("cache_clip", "cache_resnet", "study"):
    sh(f"rm -rf results/{c}"); os.makedirs(f"{DRIVE_CACHE}/{c}", exist_ok=True)
    os.makedirs("results", exist_ok=True)
    sh(f"ln -s {DRIVE_CACHE}/{c} results/{c}")
print("cache_clip warm:", len(os.listdir(f"{DRIVE_CACHE}/cache_clip")), "files")

## 3. Locate the ablation CSV\n\nThe export refuses to write anything it cannot check against the published numbers, so this file is required.

In [ ]:
CAND = [f"results/study/{CSV_NAME}", f"results/{CSV_NAME}",
        f"{DRIVE_CACHE}/{CSV_NAME}", f"{DRIVE_CACHE}/study/{CSV_NAME}",
        f"/content/drive/MyDrive/{CSV_NAME}"]
CSV_PATH = next((p for p in CAND if os.path.exists(p)), None)
if CSV_PATH is None:
    print("CSV tidak ditemukan di:", *CAND, sep="\n  ")
    print("\nUnggah", CSV_NAME, "sekarang (~23 MB):")
    from google.colab import files
    up = files.upload()
    CSV_PATH = next(iter(up))
print("CSV:", CSV_PATH, f"({os.path.getsize(CSV_PATH)/1e6:.1f} MB)")

## 4. Build the cell (cache hit expected) and recover the image paths\n\n`build_griddata` concatenates `d_cal` then `d_test` into the eval pool, and features are extracted in path order, so pool index `i` is `paths_eval[i]`. Passing `y_eval`/`g_eval` below turns that from an assumption into an assertion.

In [ ]:
import numpy as np
from study_robust_train.datasets import build_griddata, _load_bundle

def cfg_for(dataset):
    base = {"clip": {"model_name": "ViT-B-32", "pretrained": "openai", "device": "cuda",
                     "cache_dir": "results/cache_clip"},
            "resnet": {"device": "cuda", "epochs": 10, "lr": 1e-3, "batch_size": 128,
                       "cache_dir": "results/cache_resnet"}}
    base["dataset"] = {"root": os.environ["WATERBIRDS_ROOT"], "image_size": 224,
                       "n_classes": 2, "download": False}
    return base

cfg = cfg_for(DATASET)
t = time.time()
gd = build_griddata(DATASET, BACKBONE, cfg, seed=0)
print(f"[built] {BACKBONE}/{DATASET} ({(time.time()-t)/60:.1f} min), "
      f"eval pool {gd.eval_domain[0].shape}")

b = _load_bundle(DATASET, cfg, 0)          # paths only, no images decoded
cat = lambda f: list(f["d_cal"]) + list(f["d_test"])
paths_eval = cat(b.meta["paths"])
y_eval = np.concatenate([b.y["d_cal"], b.y["d_test"]]).astype(int)
g_eval = np.concatenate([b.group_id["d_cal"], b.group_id["d_test"]]).astype(int)
print("paths:", len(paths_eval), "| contoh:", paths_eval[0])

## 5. Export\n\nThree cases are selected by what they demonstrate: **rescued** (marginal drops a worst-group point, Mondrian keeps it), **widened** (both cover, Mondrian by returning a larger set), **unchanged** (a majority point both handle identically).

In [ ]:
from study_robust_train.export_example_sets import export_example_sets

man = export_example_sets(
    gd, paths_eval, CSV_PATH,
    backbone=BACKBONE, dataset=DATASET, method=METHOD, score=SCORE,
    split_seed=SPLIT_SEED, train_seed=TRAIN_SEED, rho_test=RHO_TEST,
    n_per_case=3, out_dir="results/example_sets",
    y_eval=y_eval, g_eval=g_eval)

print("\nambang:", man["thresholds"])
print("kelompok terburuk:", man["worst_group"])

## 6. Look at what came out\n\nCheck the `rescued` rows especially: the true class must be **absent** from the marginal set and **present** in the Mondrian set.

In [ ]:
from IPython.display import display
from PIL import Image

for e in man["examples"]:
    print(f"[{e['case']}] {e['file']}  benar={e['true_class']}  "
          f"grup={e['group']} ({e['attribute']}, worst={e['is_worst_group']})")
    print(f"    marginal: {e['marginal']['text']:32s} covers={e['marginal']['covers_truth']}")
    print(f"    mondrian: {e['mondrian']['text']:32s} covers={e['mondrian']['covers_truth']}")

for e in man["examples"]:
    if e["case"] == "rescued":
        display(Image.open(os.path.join("results/example_sets", e["file"])).resize((220, 220)))
        print(f"{e['true_class']}: marginal {e['marginal']['text']} -> "
              f"mondrian {e['mondrian']['text']}")
        break

## 7. Download\n\nSend the zip back — it is small (a handful of JPEGs plus a manifest).

In [ ]:
sh("cd results && zip -qr /content/example_sets.zip example_sets")
print("ukuran:", os.path.getsize("/content/example_sets.zip")/1e6, "MB")
from google.colab import files
files.download("/content/example_sets.zip")